# SMR multilink modelization


The following code is based in a PyPSA CHP multilink [example](https://pypsa.readthedocs.io/en/latest/examples/chp-fixed-heat-power-ratio.html) that demonstrates a Link component with more than one bus output (“bus2” in this case).

**The code runs correctly without "infeasible error" if there is only one output from the SMR. I am searching how to use multiple outputs in the SMR without getting the mentioned error.** 

In [8]:
import matplotlib.pyplot as plt
import pandas as pd

import pypsa

In [ ]:
# control whether the nuclear SMR_CHP is generating heat (multi-output) or not.
nuclearSMR_as_multi = True

network = pypsa.Network() # Creates empty PyPSA network object called network
network.set_snapshots(range(10)) #set time frame of simulation

network.add("Carrier", "Gas_CHP",) # define carriers (optional)
network.add("Carrier", "SMR_CHP",)
network.add("Carrier", "heat",)
network.add("Carrier","electricity",)

network.add("Bus", "Industry Electricity", carrier="electricity") # 1) Main Buses & loads
network.add("Load", "Industry Electricity Load", bus="Industry Electricity", p_set=5)
network.add("Bus", "Industry Heat", carrier="heat")
network.add("Load", "Industry Heat Load", bus="Industry Heat", p_set=3)

network.add("Bus", "uranium", carrier="SMR_CHP")                  # 2)  Fuel buses
network.add("Bus", "Spain gas", carrier="Gas_CHP")
#----------------------------------------------------------------------------------------
network.add("Store", "uranium", e_initial=1e6, e_nom=1e16, 
       bus="uranium")
network.add("Store", "Spain gas", e_initial=1e6, e_nom=1e6, # NomEnergy capacity
       bus="Spain gas")

network.add(
    "Link",
    "Gas_CHP",               
    bus0="Spain gas",
    bus1="Industry Electricity",
    bus2="Industry Heat",
    carrier="Gas_CHP",
    p_nom_extendable=True,
    marginal_cost=30,
    efficiency=1,
    efficiency2=1,
)
if nuclearSMR_as_multi:
    network.add(
        "Link",
        "SMR_CHP",
        bus0="uranium",
        bus1="Industry Electricity",
        bus2="Industry Heat", # solo comentar esta línea se va el error. 
        carrier="SMR_CHP",
        p_nom_extendable=True,
        marginal_cost=20,
        efficiency=0.35,
        efficiency2=1,
    )
else: 
    network.add(
        "Link",
        "SMR_CHP",
        bus0="uranium",
        bus1="Industry Electricity",
        # bus2="Industry Heat", # solo comentar esta línea se va el error. 
        carrier="SMR_CHP",
        p_nom_extendable=True,
        marginal_cost=20,
        efficiency=0.35,
        efficiency2=1,
    )

SyntaxError: incomplete input (1387789299.py, line 61)

In [10]:
network.optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.05s
Status: warning
Termination condition: infeasible
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: infeasible



('warning', 'infeasible')

In [11]:
# Production view
network.links_t.p0.plot(
    subplots=True, figsize=(9, 7)
)
plt.tight_layout()
print(network.links.p_nom_opt)

TypeError: no numeric data to plot

In [ ]:
print("SMR Electricity Output:", network.links_t.p1["SMR_CHP"].sum())  # -MW
print("SMR Heat Output:", network.links_t.p2["SMR_CHP"].sum())         # -MW
print("SMR Uranium Fuel Consumption:", network.links_t.p0["SMR_CHP"].sum())    #+ MW
print("Gas_CHP Electricity Output:", network.links_t.p1["Gas_CHP"].sum())
print("Gas_CHP Gas Consumption:", network.links_t.p0["Gas_CHP"].sum())

SMR Electricity Output: -20.0
SMR Heat Output: 0.0
SMR Uranium Fuel Consumption: 57.142857142857146
Gas_CHP Electricity Output: -30.0
Gas_CHP Gas Consumption: 30.0


Note that negative values for network.links_t.p1 indicate that the link is injecting power into bus1 (This aligns with PyPSA's [convention](https://pypsa.readthedocs.io/en/latest/user-guide/design.html#sign-conventions) where power injected into a bus is considered negative, while power withdrawn from a bus is positive). 

- network.links_t.p0 # energy flows through the input bus (bus0) in each link (for each snapshot) CONSUMO DE COMBUSTIBLE
- network.links_t.p1 # energy flows through the output bus1 (bus1) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)
- network.links_t.p2 # energy flows through the output bus2 (bus2) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)

<!-- Note that `negative values` for network.links_t.p1 indicate that the link is injecting power into bus1 (This aligns with PyPSA's [convention](https://pypsa.readthedocs.io/en/latest/user-guide/design.html#sign-conventions) where power injected into a bus is considered negative, while power withdrawn from a bus is positive).

Here there is a brief explanation for each of the following code lines: 
- network.links_t.p0: `energy flows through the input bus (bus0) in each link (for each snapshot) CONSUMO DE COMBUSTIBLE`
- network.links_t.p1: `energy flows through the output bus1 (bus1) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)`
- network.links_t.p2: `energy flows through the output bus2 (bus2) in each link (for each snapshot) ENERGY GENERATION (CONVERSION)` -->

# End of the fuctional example

<!-- 
Starts SMR model versión2 with network `n2`

## Note: 
About de network initialization (can be useful for some future research)

**Network Initialization**: `n1 = pypsa.Network(url)`
   - Creates a PyPSA `Network` object `n1`
   - Automatically downloads and loads the pre-configured energy system model containing:
     - Buses (energy nodes)
     - Generators (power plants)
     - Loads (energy demands)
     - Transmission lines
     - Time-dependent constraints

Key Features of This Approach:
- **Reproducibility**: Loads a standardized benchmark model
- **Time-Saving**: Avoids manual network construction (~1000+ lines of setup code)
- **Research-Ready**: Contains realistic European power system data (likely from [PyPSA-Eur](https://pypsa-eur.readthedocs.io/)) -->